In [ ]:
from google.colab import drive
# 1. DRIVE BAĞLANTISI VE VERİ ÇIKARMA
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Önce Drive'dan Colab'ın içine kopyala
!cp /content/drive/MyDrive/akilliTarimOdevi/v3-801010/content.zip /content/

# Sonra zip'i aç
!unzip -q /content/drive/MyDrive/akilliTarimOdevi/v3-801010/content.zip -d /content

In [ ]:
# Önce Drive'dan Colab'ın içine kopyala
!cp /content/drive/MyDrive/akilliTarimOdevi/v2-80515-bolunmus/akilli_tarim_test.csv /content/
!cp /content/drive/MyDrive/akilliTarimOdevi/v2-80515-bolunmus/akilli_tarim_train.csv /content/
!cp /content/drive/MyDrive/akilliTarimOdevi/v2-80515-bolunmus/akilli_tarim_val.csv /content/


# **dataseti 80 eğitim 10 val 10 test olarak bölümleme**

In [ ]:
# =================================================================
# FAZ 0 — VERİ HAZIRLIK (TEMİZ BAŞLANGIÇ)
# %80 / %10 / %10 bölme
# 3 seed: 42, 123, 7
# ChatML (SmolLM2, TinyLlama, Qwen) + Gemma4 formatı
#
# Çıktılar:
#   /content/datasets/seed_42/chatml/
#   /content/datasets/seed_42/gemma4/
#   /content/datasets/seed_123/chatml/
#   /content/datasets/seed_123/gemma4/
#   /content/datasets/seed_7/chatml/
#   /content/datasets/seed_7/gemma4/
# =================================================================

import os
import random
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_from_disk
from sklearn.model_selection import train_test_split

SEEDS = [42, 123, 7]

# -----------------------------------------------------------------
# LABEL LİSTESİ
# -----------------------------------------------------------------
LABEL_LIST = sorted([
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
])

# -----------------------------------------------------------------
# REÇETE SÖZLÜĞÜ
# -----------------------------------------------------------------
RECETE = {
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot":
        "Hasat sonrası bitki artıklarını derine gömün. Münavebe uygulayın. Tescilli fungisitler kullanılmalıdır.",
    "Corn_(maize)___Common_rust_":
        "Dayanıklı çeşitler seçilmeli, aşırı azottan kaçınılmalıdır. Uygun bir fungisit ile ilaçlama yapılmalıdır.",
    "Corn_(maize)___Northern_Leaf_Blight":
        "Hastalıklı bitki artıklarını yok edin. Çiçeklenme öncesi sistemik mantar ilaçları tercih edilmelidir.",
    "Corn_(maize)___healthy":
        "Bitkiniz tamamen sağlıklı görünmektedir. Mevcut sulama ve gübreleme programınıza devam edin.",
    "Pepper,_bell___Bacterial_spot":
        "Temiz tohum/fide kullanın. Damlama sulama tercih edin. Bakırlı preparatlar ile ilaçlama yapılmalıdır.",
    "Pepper,_bell___healthy":
        "Biber bitkinizde herhangi bir hastalık saptanmamıştır. Dengeli sulamaya devam edin.",
    "Potato___Early_blight":
        "Dengeli gübreleme ve sulama yapın. Kahverengi halkalı lekeler görüldüğünde fungisit uygulanmalıdır.",
    "Potato___Late_blight":
        "Sık dikimden kaçının. Koruyucu mildiyö ilaçları kullanılmalıdır.",
    "Potato___healthy":
        "Patates bitkiniz sağlıklı durumdadır. Boğaz doldurma ve sulama işlemlerini zamanında yapın.",
    "Tomato___Bacterial_spot":
        "Budama aletlerini dezenfekte edin. Serayı havalandırın. İlk belirtilerde bakırlı fungisitler uygulanmalıdır.",
    "Tomato___Early_blight":
        "Alt yaprakları budayarak hava akımını artırın. Yağmurlama sulamadan kaçının. Tescilli fungisitlerle ilaçlama yapın.",
    "Tomato___Late_blight":
        "Nemi düşürün, sık dikim yapmayın. Çiçeklenme veya sisli havalarda koruyucu mildiyö ilaçlaması yapın.",
    "Tomato___Leaf_Mold":
        "Seranın havalandırmasını artırarak nemi yüzde seksen beşin altına düşürün. Uygun fungisitlerle ilaçlama yapılmalıdır.",
    "Tomato___Septoria_leaf_spot":
        "Sulamayı sabah erken saatlerde yapın ki yapraklar kurusun. Belirtiler saptandığında mantar ilaçları uygulayın.",
    "Tomato___Spider_mites Two-spotted_spider_mite":
        "Yabancı ot temizliği yapın. Yaprak başına canlı sayımına göre uygun bir akarisit uygulanmalıdır.",
    "Tomato___Target_Spot":
        "Bitki mesafelerini geniş tutun, alt yaprakları budayın. Erken yanıklık ilaçları bu hastalıkta da etkilidir.",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus":
        "Doğrudan kimyasal tedavisi yoktur. Vektör olan beyazsinek ile mücadele edilmelidir. Sarı yapışkan tuzaklar kullanın.",
    "Tomato___Tomato_mosaic_virus":
        "Mekanik yolla kolay bulaşır. Dokunmadan önce eller sabunla yıkanmalıdır. Bulaşık bitkiler imha edilmelidir.",
    "Tomato___healthy":
        "Bitkiniz gayet sağlıklı gelişim göstermektedir. Düzenli sulama ve budama yapmaya devam edin.",
}

SISTEM = (
    "Sen bir bitki hastalığı teşhis asistanısın. "
    "Verilen belirti açıklamasına göre hastalığı teşhis et ve "
    "tedavi reçetesini yaz. Yanıtını 'Sınıf: <hastalık_adı>' ile başlat."
)

# -----------------------------------------------------------------
# MEVCUT HF DATASET'TEN HAM VERİ ÇIKAR
# -----------------------------------------------------------------
print("Mevcut dataset yükleniyor...")
raw = load_from_disk(
    "/content/content/akilli_tarim_hf_dataset"
)

# Tüm split'leri birleştir
tum_textler = (
    list(raw["train"]["text"]) +
    list(raw["validation"]["text"]) +
    list(raw["test"]["text"])
)

# Alpaca formatından sorgu, yanıt ve label çıkar
def alpaca_parse(text):
    try:
        sorgu = text.split("### Giriş (Belirti):")[-1]\
                    .split("### Yanıt")[0].strip()
        yanit_blok = text.split("### Yanıt (Teşhis ve Reçete):")[-1].strip()
        label = yanit_blok.split("Sınıf:")[-1].strip().split("\n")[0].strip()
        yanit = yanit_blok
    except Exception:
        sorgu, yanit, label = text, "", "Bilinmeyen"
    return sorgu, yanit, label

kayitlar = []
for text in tum_textler:
    sorgu, yanit, label = alpaca_parse(text)
    if label in LABEL_LIST:
        kayitlar.append({"Sorgu": sorgu, "Yanit": yanit, "Label": label})

df = pd.DataFrame(kayitlar).drop_duplicates(subset=["Sorgu"]).reset_index(drop=True)
print(f"Toplam benzersiz kayıt: {len(df)}")
print(f"Sınıf dağılımı:\n{df['Label'].value_counts()}")

# -----------------------------------------------------------------
# FORMAT FONKSİYONLARI
# -----------------------------------------------------------------
def chatml_formatla(row):
    yanit = f"Sınıf: {row['Label']}\n{RECETE.get(row['Label'], '')}"
    return (
        f"<|im_start|>system\n{SISTEM}<|im_end|>\n"
        f"<|im_start|>user\nBelirti: {row['Sorgu']}<|im_end|>\n"
        f"<|im_start|>assistant\n{yanit}<|im_end|>"
    )

def gemma4_formatla(row):
    yanit = f"Sınıf: {row['Label']}\n{RECETE.get(row['Label'], '')}"
    return (
        f"<start_of_turn>user\n{SISTEM}\n\nBelirti: {row['Sorgu']}<end_of_turn>\n"
        f"<start_of_turn>model\n{yanit}<end_of_turn>"
    )

# -----------------------------------------------------------------
# 3 SEED İÇİN BÖLE, FORMATLA VE KAYDET
# -----------------------------------------------------------------
for seed in SEEDS:
    random.seed(seed)
    np.random.seed(seed)

    print(f"\n{'='*50}")
    print(f"SEED {seed} işleniyor...")
    print(f"{'='*50}")

    # %80 / %10 / %10 stratified bölme
    df_train, df_temp = train_test_split(
        df, test_size=0.20, random_state=seed, stratify=df["Label"]
    )
    df_val, df_test = train_test_split(
        df_temp, test_size=0.50, random_state=seed, stratify=df_temp["Label"]
    )

    print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
    print(f"Oranlar: %{len(df_train)/len(df)*100:.1f} / "
          f"%{len(df_val)/len(df)*100:.1f} / "
          f"%{len(df_test)/len(df)*100:.1f}")

    # Test CSV'sini kaydet (inference için)
    base = f"/content/datasets/seed_{seed}"
    os.makedirs(base, exist_ok=True)
    df_test[["Sorgu", "Yanit", "Label"]].to_csv(
        f"{base}/test.csv", index=False
    )

    for fmt_adi, fmt_fn in [("chatml", chatml_formatla),
                             ("gemma4", gemma4_formatla)]:
        for split_adi, split_df in [("train", df_train),
                                     ("validation", df_val),
                                     ("test", df_test)]:
            split_df = split_df.copy()
            split_df["text"] = split_df.apply(fmt_fn, axis=1)

        dataset = DatasetDict({
            "train":      Dataset.from_pandas(
                df_train.copy().assign(
                    text=df_train.apply(fmt_fn, axis=1)
                )[["text", "Label"]].reset_index(drop=True)
            ),
            "validation": Dataset.from_pandas(
                df_val.copy().assign(
                    text=df_val.apply(fmt_fn, axis=1)
                )[["text", "Label"]].reset_index(drop=True)
            ),
            "test":       Dataset.from_pandas(
                df_test.copy().assign(
                    text=df_test.apply(fmt_fn, axis=1)
                )[["text", "Label"]].reset_index(drop=True)
            ),
        })

        kayit_yolu = f"{base}/{fmt_adi}"
        dataset.save_to_disk(kayit_yolu)
        print(f"  Kaydedildi: {kayit_yolu}")

    # Örnek göster (sadece seed 42 için)
    if seed == 42:
        ornek = df_train.iloc[0]
        print(f"\n--- ChatML Örnek (seed 42) ---")
        print(chatml_formatla(ornek)[:400])
        print(f"\n--- Gemma4 Örnek (seed 42) ---")
        print(gemma4_formatla(ornek)[:400])

print(f"""
✅ VERİ HAZIRLIK TAMAMLANDI!

Oluşturulan klasörler:
  /content/datasets/seed_42/chatml/   → SmolLM2, TinyLlama, Qwen
  /content/datasets/seed_42/gemma4/   → Gemma4 E2B
  /content/datasets/seed_123/chatml/
  /content/datasets/seed_123/gemma4/
  /content/datasets/seed_7/chatml/
  /content/datasets/seed_7/gemma4/

Her klasörde: train / validation / test split'leri
Her seed için: test.csv (inference'da kullanılacak)

Bölme oranı: %80 / %10 / %10 (stratified)
Seed'ler   : 42, 123, 7
""")

Mevcut dataset yükleniyor...
Toplam benzersiz kayıt: 1461
Sınıf dağılımı:
Label
Tomato___Tomato_Yellow_Leaf_Curl_Virus                84
Tomato___Spider_mites Two-spotted_spider_mite         84
Tomato___Tomato_mosaic_virus                          81
Tomato___Septoria_leaf_spot                           81
Tomato___healthy                                      80
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot    80
Potato___healthy                                      80
Pepper,_bell___healthy                                79
Corn_(maize)___Northern_Leaf_Blight                   77
Potato___Early_blight                                 77
Corn_(maize)___healthy                                77
Pepper,_bell___Bacterial_spot                         76
Potato___Late_blight                                  76
Tomato___Leaf_Mold                                    76
Tomato___Bacterial_spot                               74
Tomato___Early_blight                                 70
Tomato__

Saving the dataset (0/1 shards):   0%|          | 0/1168 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/146 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/147 [00:00<?, ? examples/s]

  Kaydedildi: /content/datasets/seed_42/chatml


...

  Kaydedildi: /content/datasets/seed_7/chatml


Saving the dataset (0/1 shards):   0%|          | 0/1168 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/146 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/147 [00:00<?, ? examples/s]

  Kaydedildi: /content/datasets/seed_7/gemma4

✅ VERİ HAZIRLIK TAMAMLANDI!

Oluşturulan klasörler:
  /content/datasets/seed_42/chatml/   → SmolLM2, TinyLlama, Qwen
  /content/datasets/seed_42/gemma4/   → Gemma4 E2B
  /content/datasets/seed_123/chatml/
  /content/datasets/seed_123/gemma4/
  /content/datasets/seed_7/chatml/
  /content/datasets/seed_7/gemma4/

Her klasörde: train / validation / test split'leri
Her seed için: test.csv (inference'da kullanılacak)

Bölme oranı: %80 / %10 / %10 (stratified)
Seed'ler   : 42, 123, 7



# **zero-shot baseline **

In [ ]:
# =================================================================
# FAZ 0 — ZERO-SHOT BASELINE
# 4 model × 3 seed → ort ± std tablosu
#
# Metrikler: Accuracy, Macro-F1, Weighted-F1, ROC-AUC,
#            ROUGE-1, ROUGE-L, Inference (ms/örnek),
#            GPU bellek (GB)
#
# Çıktı: /content/zeroshot_sonuclar/
# =================================================================

import subprocess
subprocess.run(["pip", "install", "-q", "trl", "peft", "transformers",
                "bitsandbytes", "accelerate", "rouge-score", "-U"], check=True)

import os, json, time, gc
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from rouge_score import rouge_scorer

SAVE_PATH = "/content/zeroshot_sonuclar"
os.makedirs(SAVE_PATH, exist_ok=True)

SEEDS      = [42, 123, 7]
BATCH_SIZE = 4
MAX_NEW_TOKENS = 25

# -----------------------------------------------------------------
# LABEL LİSTESİ
# -----------------------------------------------------------------
LABEL_LIST = sorted([
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
])
label2id   = {l: i for i, l in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

# -----------------------------------------------------------------
# 4 MODEL TANIMI
# -----------------------------------------------------------------
MODELLER = [
    {
        "ad":       "SmolLM2-360M",
        "model_id": "HuggingFaceTB/SmolLM2-360M-Instruct",
        "format":   "chatml",
        "params":   "360M",
    },
    {
        "ad":       "TinyLlama-1.1B",
        "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "format":   "chatml",
        "params":   "1.1B",
    },
    {
        "ad":       "Qwen2.5-1.5B",
        "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
        "format":   "chatml",
        "params":   "1.5B",
    },
    {
        "ad":       "Gemma4-E2B",
        "model_id": "google/gemma-4-E2B-it",
        "format":   "gemma4",
        "params":   "~2B",
    },
]

SISTEM = (
    "Sen bir bitki hastalığı teşhis asistanısın. "
    "Verilen belirti açıklamasına göre hastalığı teşhis et ve "
    "tedavi reçetesini yaz. Yanıtını 'Sınıf: <hastalık_adı>' ile başlat."
)

def build_prompt(sorgu, fmt):
    if fmt == "gemma4":
        return (
            f"<start_of_turn>user\n{SISTEM}\n\n"
            f"Belirti: {sorgu}<end_of_turn>\n"
            f"<start_of_turn>model\nSınıf:"
        )
    return (
        f"<|im_start|>system\n{SISTEM}<|im_end|>\n"
        f"<|im_start|>user\nBelirti: {sorgu}<|im_end|>\n"
        f"<|im_start|>assistant\nSınıf:"
    )

def parse_label(text):
    if "Sınıf:" not in text:
        return "Bilinmeyen"
    return text.split("Sınıf:")[-1].strip().split("\n")[0].strip()

# -----------------------------------------------------------------
# TEST ÇİFTLERİNİ ÇIKAR
# -----------------------------------------------------------------
def test_ciftleri_al(seed, fmt):
    ds = load_from_disk(f"/content/datasets/seed_{seed}/{fmt}")
    pairs = []
    for row in ds["test"]:
        text = row["text"]
        try:
            if fmt == "chatml":
                sorgu = text.split("<|im_start|>user\nBelirti:")[-1]\
                            .split("<|im_end|>")[0].strip()
                yanit = text.split("<|im_start|>assistant\n")[-1]\
                            .split("<|im_end|>")[0].strip()
            else:
                sorgu = text.split("Belirti:")[-1].split("<end_of_turn>")[0].strip()
                yanit = text.split("<start_of_turn>model\n")[-1]\
                            .split("<end_of_turn>")[0].strip()
            label = yanit.split("Sınıf:")[-1].strip().split("\n")[0].strip()
        except Exception:
            sorgu, yanit, label = text, "", "Bilinmeyen"
        pairs.append((sorgu, yanit, label))
    return pairs

# -----------------------------------------------------------------
# GPU BELLEK ÖLÇÜMÜ
# -----------------------------------------------------------------
def gpu_bellek_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return 0.0

# -----------------------------------------------------------------
# METRİK HESAPLAMA
# -----------------------------------------------------------------
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

def metrik_hesapla(gercek_ids, tahmin_ids, tahmin_vecs,
                   gercek_yanitlar, tahmin_yanitlar):
    acc         = accuracy_score(gercek_ids, tahmin_ids)
    macro_f1    = f1_score(gercek_ids, tahmin_ids, average="macro",    zero_division=0)
    weighted_f1 = f1_score(gercek_ids, tahmin_ids, average="weighted", zero_division=0)
    gercek_bin  = label_binarize(gercek_ids, classes=list(range(NUM_LABELS)))
    try:
        roc_auc = roc_auc_score(gercek_bin, np.array(tahmin_vecs),
                                average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = float("nan")

    # ROUGE (reçete kalitesi)
    r1_scores, rL_scores = [], []
    for ref, hyp in zip(gercek_yanitlar, tahmin_yanitlar):
        if ref and hyp:
            s = rouge.score(ref, hyp)
            r1_scores.append(s["rouge1"].fmeasure)
            rL_scores.append(s["rougeL"].fmeasure)
    rouge1 = float(np.mean(r1_scores)) if r1_scores else 0.0
    rougeL = float(np.mean(rL_scores)) if rL_scores else 0.0

    return acc, macro_f1, weighted_f1, roc_auc, rouge1, rougeL

# -----------------------------------------------------------------
# ANA DÖNGÜ
# -----------------------------------------------------------------
tum_sonuclar = []  # her satır: model × seed

for model_cfg in MODELLER:
    print(f"\n{'='*60}")
    print(f"MODEL: {model_cfg['ad']}  ({model_cfg['model_id']})")
    print(f"{'='*60}")

    # Modeli bir kez yükle, 3 seed için kullan
    torch.cuda.reset_peak_memory_stats()

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["model_id"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_cfg["model_id"],
        quantization_config=bnb,
        device_map="auto",
    )
    model.eval()

    gpu_gb = gpu_bellek_gb()

    for seed in SEEDS:
        print(f"\n  Seed {seed} çalışıyor...")

        pairs = test_ciftleri_al(seed, model_cfg["format"])
        gercek_ids, tahmin_ids, tahmin_vecs = [], [], []
        gercek_yanitlar, tahmin_yanitlar    = [], []
        inf_sureler = []

        for i in range(0, len(pairs), BATCH_SIZE):
            batch     = pairs[i: i + BATCH_SIZE]
            sorgular  = [p[0] for p in batch]
            gercekler = [p[1] for p in batch]   # reçete metni
            labellar  = [p[2] for p in batch]

            prompts = [build_prompt(s, model_cfg["format"]) for s in sorgular]
            inputs  = tokenizer(
                prompts, return_tensors="pt",
                padding=True, truncation=True, max_length=256
            ).to("cuda" if torch.cuda.is_available() else "cpu")

            t0 = time.time()
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )
            inf_sureler.append((time.time() - t0) / len(batch))

            for j, out in enumerate(outputs):
                decoded  = tokenizer.decode(out, skip_special_tokens=True)
                tahmin   = parse_label(decoded)
                g_id = label2id.get(labellar[j], 0)
                t_id = label2id.get(tahmin,      0)
                vec  = np.zeros(NUM_LABELS); vec[t_id] = 1.0

                gercek_ids.append(g_id)
                tahmin_ids.append(t_id)
                tahmin_vecs.append(vec)
                gercek_yanitlar.append(gercekler[j])
                tahmin_yanitlar.append(decoded)

        acc, macro_f1, weighted_f1, roc_auc, rouge1, rougeL = metrik_hesapla(
            gercek_ids, tahmin_ids, tahmin_vecs,
            gercek_yanitlar, tahmin_yanitlar
        )
        inf_ms = np.mean(inf_sureler) * 1000

        satir = {
            "Model":        model_cfg["ad"],
            "Parametre":    model_cfg["params"],
            "FT_Yontemi":   "Zero-shot",
            "Seed":         seed,
            "Accuracy":     round(acc,         4),
            "Macro_F1":     round(macro_f1,    4),
            "Weighted_F1":  round(weighted_f1, 4),
            "ROC_AUC":      round(roc_auc,     4) if not np.isnan(roc_auc) else float("nan"),
            "ROUGE_1":      round(rouge1,      4),
            "ROUGE_L":      round(rougeL,      4),
            "Inf_ms":       round(inf_ms,      1),
            "GPU_GB":       round(gpu_gb,      2),
        }
        tum_sonuclar.append(satir)
        print(f"    Acc={acc:.4f} | F1={macro_f1:.4f} | "
              f"ROUGE-1={rouge1:.4f} | GPU={gpu_gb:.2f}GB | "
              f"Inf={inf_ms:.1f}ms")

    # Modeli bellekten temizle
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU belleği temizlendi.")

# -----------------------------------------------------------------
# SONUÇLARI KAYDET — ham tablo
# -----------------------------------------------------------------
df_raw = pd.DataFrame(tum_sonuclar)
df_raw.to_csv(f"{SAVE_PATH}/zeroshot_ham.csv", index=False)

# -----------------------------------------------------------------
# ORT ± STD TABLOSU
# -----------------------------------------------------------------
metrik_sutunlar = ["Accuracy","Macro_F1","Weighted_F1",
                   "ROC_AUC","ROUGE_1","ROUGE_L","Inf_ms","GPU_GB"]

satirlar = []
for model_ad in df_raw["Model"].unique():
    alt = df_raw[df_raw["Model"] == model_ad]
    satir = {
        "Model":     model_ad,
        "Parametre": alt["Parametre"].iloc[0],
        "FT":        "Zero-shot",
    }
    for m in metrik_sutunlar:
        ort = alt[m].mean()
        std = alt[m].std()
        satir[f"{m}_ort"] = round(ort, 4)
        satir[f"{m}_std"] = round(std, 4)
        satir[f"{m}"]     = f"{ort:.4f} ± {std:.4f}"
    satirlar.append(satir)

df_ozet = pd.DataFrame(satirlar)
df_ozet.to_csv(f"{SAVE_PATH}/zeroshot_ozet.csv", index=False)

# -----------------------------------------------------------------
# GRAFİK — 4 model × 3 metrik
# -----------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrikler = ["Accuracy", "Macro_F1", "ROUGE_1"]
renkler   = ["#7F77DD", "#1D9E75", "#BA7517", "#D85A30"]
m_adlari  = df_raw["Model"].unique()

for ax, metrik in zip(axes, metrikler):
    for idx, model_ad in enumerate(m_adlari):
        alt    = df_raw[df_raw["Model"] == model_ad]
        degerler = alt[metrik].values
        x = idx
        ax.bar(x, degerler.mean(), color=renkler[idx],
               width=0.5, edgecolor="white", label=model_ad)
        ax.errorbar(x, degerler.mean(), yerr=degerler.std(),
                    fmt="none", color="black", capsize=4, linewidth=1.5)
    ax.set_title(f"Zero-shot {metrik}", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_xticks(range(len(m_adlari)))
    ax.set_xticklabels([m.replace("-", "\n") for m in m_adlari],
                       fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Zero-shot Baseline — 4 Model × 3 Seed (ort ± std)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/zeroshot_grafik.png", dpi=150, bbox_inches="tight")
plt.close()

# -----------------------------------------------------------------
# SONUÇ YAZDIR
# -----------------------------------------------------------------
print(f"\n{'='*60}")
print("ZERO-SHOT BASELINE SONUÇLARI (ort ± std)")
print(f"{'='*60}")
for _, row in df_ozet.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  Accuracy  : {row['Accuracy']}")
    print(f"  Macro F1  : {row['Macro_F1']}")
    print(f"  ROUGE-1   : {row['ROUGE_1']}")
    print(f"  ROUGE-L   : {row['ROUGE_L']}")
    print(f"  Inf (ms)  : {row['Inf_ms']}")
    print(f"  GPU (GB)  : {row['GPU_GB']}")

print(f"""
✅ FAZ 0 TAMAMLANDI!
   /content/zeroshot_sonuclar/zeroshot_ham.csv   → 4 model × 3 seed ham veriler
   /content/zeroshot_sonuclar/zeroshot_ozet.csv  → ort ± std tablosu
   /content/zeroshot_sonuclar/zeroshot_grafik.png
""")


MODEL: SmolLM2-360M  (HuggingFaceTB/SmolLM2-360M-Instruct)


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]


  Seed 42 çalışıyor...
    Acc=0.0544 | F1=0.0054 | ROUGE-1=0.1503 | GPU=6.40GB | Inf=668.6ms

  Seed 123 çalışıyor...
    Acc=0.0544 | F1=0.0054 | ROUGE-1=0.1519 | GPU=6.40GB | Inf=648.9ms

  Seed 7 çalışıyor...
    Acc=0.0544 | F1=0.0054 | ROUGE-1=0.1517 | GPU=6.40GB | Inf=692.8ms
  GPU belleği temizlendi.

ZERO-SHOT BASELINE SONUÇLARI (ort ± std)

SmolLM2-360M:
  Accuracy  : 0.0544 ± 0.0000
  Macro F1  : 0.0054 ± 0.0000
  ROUGE-1   : 0.1608 ± 0.0016
  ROUGE-L   : 0.1224 ± 0.0010
  Inf (ms)  : 484.6000 ± 6.6430
  GPU (GB)  : 0.2600 ± 0.0000

TinyLlama-1.1B:
  Accuracy  : 0.0544 ± 0.0000
  Macro F1  : 0.0054 ± 0.0000
  ROUGE-1   : 0.1450 ± 0.0012
  ROUGE-L   : 0.1100 ± 0.0004
  Inf (ms)  : 369.2333 ± 1.2055
  GPU (GB)  : 0.7900 ± 0.0000

Qwen2.5-1.5B:
  Accuracy  : 0.0544 ± 0.0000
  Macro F1  : 0.0054 ± 0.0000
  ROUGE-1   : 0.1654 ± 0.0014
  ROUGE-L   : 0.1247 ± 0.0019
  Inf (ms)  : 398.5000 ± 14.7625
  GPU (GB)  : 1.1600 ± 0.0000

Gemma4-E2B:
  Accuracy  : 0.0544 ± 0.0000
  Macro F1

# **smollm2**

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "trl", "peft", "transformers",
                "bitsandbytes", "accelerate", "rouge-score", "-U"], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'trl', 'peft', 'transformers', 'bitsandbytes', 'accelerate', 'rouge-score', '-U'], returncode=0)

In [ ]:
# =================================================================
# FAZ 1 — SmolLM2-360M LoRA Fine-tuning
# 3 seed: 42, 123, 7
# Metrikler: Acc, Macro-F1, Weighted-F1, ROC-AUC,
#            ROUGE-1, ROUGE-L, Inf (ms), GPU (GB)
# Çıktı: /content/smollm2_output/
# =================================================================

import subprocess
subprocess.run(["pip", "install", "-q", "trl", "peft", "transformers",
                "bitsandbytes", "accelerate", "rouge-score", "-U"], check=True)

import os, json, time, gc, random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
from rouge_score import rouge_scorer

# -----------------------------------------------------------------
# AYARLAR
# -----------------------------------------------------------------
MODEL_NAME  = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_LABEL = "SmolLM2-360M"
MODEL_PARAMS = "360M"
DATASET_FMT  = "chatml"
SEEDS        = [42, 123, 7]
SAVE_PATH    = "/content/smollm2_output"
os.makedirs(SAVE_PATH, exist_ok=True)

login(token=userdata.get("HF_TOKEN"))

# -----------------------------------------------------------------
# LABEL LİSTESİ
# -----------------------------------------------------------------
LABEL_LIST = sorted([
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
])
label2id    = {l: i for i, l in enumerate(LABEL_LIST)}
NUM_LABELS  = len(LABEL_LIST)
short_labels = [l.split("___")[-1][:18] for l in LABEL_LIST]

rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

# -----------------------------------------------------------------
# YARDIMCI FONKSİYONLAR
# -----------------------------------------------------------------
def build_prompt(sorgu):
    return (
        f"<|im_start|>system\nSen bir bitki hastalığı teşhis asistanısın. "
        f"Yanıtını 'Sınıf: <hastalık_adı>' ile başlat.<|im_end|>\n"
        f"<|im_start|>user\nBelirti: {sorgu}<|im_end|>\n"
        f"<|im_start|>assistant\nSınıf:"
    )

def parse_label(text):
    if "Sınıf:" not in text:
        return "Bilinmeyen"
    return text.split("Sınıf:")[-1].strip().split("\n")[0].strip()

def test_ciftleri_al(seed):
    ds = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")
    pairs = []
    for row in ds["test"]:
        text = row["text"]
        try:
            sorgu = text.split("<|im_start|>user\nBelirti:")[-1]\
                        .split("<|im_end|>")[0].strip()
            yanit = text.split("<|im_start|>assistant\n")[-1]\
                        .split("<|im_end|>")[0].strip()
            label = yanit.split("Sınıf:")[-1].strip().split("\n")[0].strip()
        except Exception:
            sorgu, yanit, label = text, "", "Bilinmeyen"
        pairs.append((sorgu, yanit, label))
    return pairs

def metrik_hesapla(gercek_ids, tahmin_ids, tahmin_vecs,
                   gercek_yanitlar, tahmin_yanitlar):
    acc         = accuracy_score(gercek_ids, tahmin_ids)
    macro_f1    = f1_score(gercek_ids, tahmin_ids, average="macro",    zero_division=0)
    weighted_f1 = f1_score(gercek_ids, tahmin_ids, average="weighted", zero_division=0)
    f1_per_cls  = f1_score(gercek_ids, tahmin_ids, average=None,
                            zero_division=0, labels=list(range(NUM_LABELS)))
    gercek_bin  = label_binarize(gercek_ids, classes=list(range(NUM_LABELS)))
    try:
        roc_auc = roc_auc_score(gercek_bin, np.array(tahmin_vecs),
                                 average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = float("nan")
    r1, rL = [], []
    for ref, hyp in zip(gercek_yanitlar, tahmin_yanitlar):
        if ref and hyp:
            s = rouge.score(ref, hyp)
            r1.append(s["rouge1"].fmeasure)
            rL.append(s["rougeL"].fmeasure)
    return (acc, macro_f1, weighted_f1, roc_auc,
            float(np.mean(r1)) if r1 else 0.0,
            float(np.mean(rL)) if rL else 0.0,
            f1_per_cls)

def gpu_bellek_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return 0.0

# -----------------------------------------------------------------
# LORA KONFİGÜRASYONU
# -----------------------------------------------------------------
def lora_config_olustur():
    return LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                         "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM
    )

# -----------------------------------------------------------------
# ANA DÖNGÜ — 3 SEED
# -----------------------------------------------------------------
tum_sonuclar  = []
tum_log       = {}
tum_f1_per_cls = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"{MODEL_LABEL} — SEED {seed}")
    print(f"{'='*60}")

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.reset_peak_memory_stats()

    dataset = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")

    # Model yükle
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    # Eğitim
    egitim_bas = time.time()
    sft_config = SFTConfig(
        max_length=256, dataset_text_field="text",
        output_dir=f"{SAVE_PATH}/seed_{seed}/checkpoints",
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=4, per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=5, weight_decay=0.01,
        load_best_model_at_end=True, logging_steps=10,
        fp16=False, bf16=True, report_to="none", seed=seed,
    )
    trainer = SFTTrainer(
        model=model, args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config_olustur(),
    )
    trainer.train()
    egitim_sure = (time.time() - egitim_bas) / 60
    log_history  = trainer.state.log_history
    tum_log[f"seed_{seed}"] = log_history

    # Model kaydet
    seed_path = f"{SAVE_PATH}/seed_{seed}"
    os.makedirs(seed_path, exist_ok=True)
    trainer.save_model(f"{seed_path}/best_model")

    # Loss grafikleri
    train_steps  = [l["step"] for l in log_history if "loss" in l]
    train_losses = [l["loss"]      for l in log_history if "loss" in l]
    val_steps    = [l["step"]      for l in log_history if "eval_loss" in l]
    val_losses   = [l["eval_loss"] for l in log_history if "eval_loss" in l]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_steps, train_losses, label="Train Loss",
            color="#7F77DD", lw=2)
    if val_losses:
        ax.plot(val_steps, val_losses, label="Val Loss",
                color="#FF5722", lw=2, marker="o", ms=5)
    ax.set_title(f"{MODEL_LABEL} — Loss (seed {seed})",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Adım"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{seed_path}/loss_egrisi.png", dpi=150)
    plt.close()

    # Test inference
    model.eval()
    tokenizer.padding_side = "left"
    pairs = test_ciftleri_al(seed)

    gercek_ids, tahmin_ids, tahmin_vecs = [], [], []
    gercek_yanitlar, tahmin_yanitlar    = [], []
    inf_sureler = []

    for i in range(0, len(pairs), 8):
        batch     = pairs[i: i + 8]
        prompts   = [build_prompt(p[0]) for p in batch]
        gercekler = [p[1] for p in batch]
        labellar  = [p[2] for p in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=256
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=25,
                do_sample=False, pad_token_id=tokenizer.pad_token_id
            )
        inf_sureler.append((time.time() - t0) / len(batch))

        for j, out in enumerate(outputs):
            decoded = tokenizer.decode(out, skip_special_tokens=True)
            tahmin  = parse_label(decoded)
            g_id = label2id.get(labellar[j], 0)
            t_id = label2id.get(tahmin,      0)
            vec  = np.zeros(NUM_LABELS); vec[t_id] = 1.0
            gercek_ids.append(g_id); tahmin_ids.append(t_id)
            tahmin_vecs.append(vec)
            gercek_yanitlar.append(gercekler[j])
            tahmin_yanitlar.append(decoded)

    inf_ms  = np.mean(inf_sureler) * 1000
    gpu_gb  = gpu_bellek_gb()
    model_mb = sum(p.numel() * p.element_size()
                   for p in model.parameters()) / 1024**2

    (acc, macro_f1, weighted_f1, roc_auc,
     rouge1, rougeL, f1_per_cls) = metrik_hesapla(
        gercek_ids, tahmin_ids, tahmin_vecs,
        gercek_yanitlar, tahmin_yanitlar
    )
    tum_f1_per_cls.append(f1_per_cls)

    satir = {
        "Model":        MODEL_LABEL,
        "Parametre":    MODEL_PARAMS,
        "FT_Yontemi":   "LoRA",
        "Seed":         seed,
        "Accuracy":     round(acc,         4),
        "Macro_F1":     round(macro_f1,    4),
        "Weighted_F1":  round(weighted_f1, 4),
        "ROC_AUC":      round(roc_auc,     4) if not np.isnan(roc_auc) else float("nan"),
        "ROUGE_1":      round(rouge1,      4),
        "ROUGE_L":      round(rougeL,      4),
        "Egitim_dk":    round(egitim_sure, 1),
        "Inf_ms":       round(inf_ms,      1),
        "GPU_GB":       round(gpu_gb,      2),
        "Model_MB":     round(model_mb,    0),
    }
    tum_sonuclar.append(satir)

    print(f"  Acc={acc:.4f} | F1={macro_f1:.4f} | "
          f"ROUGE-1={rouge1:.4f} | GPU={gpu_gb:.2f}GB | "
          f"Egitim={egitim_sure:.1f}dk")

    # Confusion matrix (sadece seed 42)
    if seed == 42:
        cm = confusion_matrix(gercek_ids, tahmin_ids,
                              labels=list(range(NUM_LABELS)))
        fig, ax = plt.subplots(figsize=(16, 14))
        ConfusionMatrixDisplay(cm, display_labels=short_labels).plot(
            ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
        ax.set_title(f"{MODEL_LABEL} — Confusion Matrix (seed 42)",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"{SAVE_PATH}/confusion_matrix.png", dpi=130)
        plt.close()

    # Modeli temizle
    del model, tokenizer, trainer
    gc.collect(); torch.cuda.empty_cache()

# -----------------------------------------------------------------
# ORT ± STD TABLOSU
# -----------------------------------------------------------------
df_raw  = pd.DataFrame(tum_sonuclar)
df_raw.to_csv(f"{SAVE_PATH}/ham_sonuclar.csv", index=False)

metrik_sutunlar = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC",
                   "ROUGE_1","ROUGE_L","Egitim_dk","Inf_ms","GPU_GB"]
ozet = {"Model": MODEL_LABEL, "Parametre": MODEL_PARAMS, "FT": "LoRA"}
for m in metrik_sutunlar:
    ort = df_raw[m].mean()
    std = df_raw[m].std()
    ozet[m] = f"{ort:.4f} ± {std:.4f}"
    ozet[f"{m}_ort"] = round(ort, 4)
    ozet[f"{m}_std"] = round(std, 4)

pd.DataFrame([ozet]).to_csv(f"{SAVE_PATH}/ozet_sonuclar.csv", index=False)

# -----------------------------------------------------------------
# GRAFİKLER
# -----------------------------------------------------------------
# Test metrikleri (ort ± std)
fig, ax = plt.subplots(figsize=(10, 5))
m_labels = ["Accuracy","Macro F1","Weighted F1","ROC-AUC","ROUGE-1","ROUGE-L"]
m_keys   = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC","ROUGE_1","ROUGE_L"]
m_vals   = [ozet[f"{k}_ort"] for k in m_keys]
m_stds   = [ozet[f"{k}_std"] for k in m_keys]
colors   = ["#4CAF50","#2196F3","#FF9800","#9C27B0","#E91E63","#00BCD4"]
bars = ax.bar(m_labels, m_vals, color=colors, width=0.5, edgecolor="white")
ax.errorbar(range(len(m_vals)), m_vals, yerr=m_stds,
            fmt="none", color="black", capsize=5, linewidth=1.5)
ax.set_ylim(0, 1.2); ax.set_ylabel("Skor")
ax.set_title(f"{MODEL_LABEL} — Test Metrikleri (ort ± std, 3 seed)",
             fontsize=13, fontweight="bold")
for bar, val in zip(bars, m_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/test_metrikleri.png", dpi=150)
plt.close()

# Sınıf bazlı F1 (3 seed ortalaması)
f1_ort = np.mean(tum_f1_per_cls, axis=0)
f1_std = np.std(tum_f1_per_cls, axis=0)
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(NUM_LABELS)
ax.barh(y, f1_ort, xerr=f1_std, color="#7F77DD",
        edgecolor="white", capsize=3)
ax.set_yticks(y); ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("F1 Skoru")
ax.set_title(f"{MODEL_LABEL} — Sınıf Bazlı F1 (ort ± std)",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.2)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/sinif_f1.png", dpi=150)
plt.close()

# Log kaydet
with open(f"{SAVE_PATH}/history_log.json", "w") as f:
    json.dump(tum_log, f, indent=2)

import shutil
shutil.make_archive("/content/smollm2_output_zip", "zip", SAVE_PATH)

print(f"""
✅ FAZ 1 TAMAMLANDI — {MODEL_LABEL}
   Accuracy   : {ozet['Accuracy']}
   Macro F1   : {ozet['Macro_F1']}
   ROUGE-1    : {ozet['ROUGE_1']}
   Eğitim     : {ozet['Egitim_dk']} dk
   GPU        : {ozet['GPU_GB']} GB

   Dosyalar: /content/smollm2_output/
   ZIP      : /content/smollm2_output_zip.zip
""")


SmolLM2-360M — SEED 42


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.587701,0.455738
2,0.133116,0.119025
3,0.085006,0.083823
4,0.068674,0.072679
5,0.063344,0.071029


  Acc=0.5646 | F1=0.5252 | ROUGE-1=0.2992 | GPU=1.31GB | Egitim=15.9dk

SmolLM2-360M — SEED 123


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Tokenizing train dataset:   0%|          | 0/1168 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.612560,0.463189
2,0.128926,0.114477
3,0.084082,0.082289
4,0.068627,0.072256
5,0.063043,0.069938


  Acc=0.5918 | F1=0.5660 | ROUGE-1=0.3053 | GPU=1.31GB | Egitim=16.1dk

✅ FAZ 1 TAMAMLANDI — SmolLM2-360M
   Accuracy   : 0.5827 ± 0.0157
   Macro F1   : 0.5509 ± 0.0223
   ROUGE-1    : 0.2994 ± 0.0058
   Eğitim     : 15.9667 ± 0.1155 dk
   GPU        : 1.3100 ± 0.0000 GB

   Dosyalar: /content/smollm2_output/
   ZIP      : /content/smollm2_output_zip.zip



# **tiny**

In [ ]:
# =================================================================
# FAZ 1 — SmolLM2-360M LoRA Fine-tuning
# 3 seed: 42, 123, 7
# Metrikler: Acc, Macro-F1, Weighted-F1, ROC-AUC,
#            ROUGE-1, ROUGE-L, Inf (ms), GPU (GB)
# Çıktı: /content/smollm2_output/
# =================================================================

import subprocess
subprocess.run(["pip", "install", "-q", "trl", "peft", "transformers",
                "bitsandbytes", "accelerate", "rouge-score", "-U"], check=True)

import os, json, time, gc, random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
from rouge_score import rouge_scorer

# -----------------------------------------------------------------
# AYARLAR
# -----------------------------------------------------------------
MODEL_NAME  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_LABEL = "TinyLlama-1.1B"
MODEL_PARAMS = "1.1B"
DATASET_FMT  = "chatml"
SEEDS        = [42, 123, 7]
SAVE_PATH    = "/content/tinyllama_output"
os.makedirs(SAVE_PATH, exist_ok=True)

login(token=userdata.get("HF_TOKEN"))

# -----------------------------------------------------------------
# LABEL LİSTESİ
# -----------------------------------------------------------------
LABEL_LIST = sorted([
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
])
label2id    = {l: i for i, l in enumerate(LABEL_LIST)}
NUM_LABELS  = len(LABEL_LIST)
short_labels = [l.split("___")[-1][:18] for l in LABEL_LIST]

rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

# -----------------------------------------------------------------
# YARDIMCI FONKSİYONLAR
# -----------------------------------------------------------------
def build_prompt(sorgu):
    return (
        f"<|im_start|>system\nSen bir bitki hastalığı teşhis asistanısın. "
        f"Yanıtını 'Sınıf: <hastalık_adı>' ile başlat.<|im_end|>\n"
        f"<|im_start|>user\nBelirti: {sorgu}<|im_end|>\n"
        f"<|im_start|>assistant\nSınıf:"
    )

def parse_label(text):
    if "Sınıf:" not in text:
        return "Bilinmeyen"
    return text.split("Sınıf:")[-1].strip().split("\n")[0].strip()

def test_ciftleri_al(seed):
    ds = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")
    pairs = []
    for row in ds["test"]:
        text = row["text"]
        try:
            sorgu = text.split("<|im_start|>user\nBelirti:")[-1]\
                        .split("<|im_end|>")[0].strip()
            yanit = text.split("<|im_start|>assistant\n")[-1]\
                        .split("<|im_end|>")[0].strip()
            label = yanit.split("Sınıf:")[-1].strip().split("\n")[0].strip()
        except Exception:
            sorgu, yanit, label = text, "", "Bilinmeyen"
        pairs.append((sorgu, yanit, label))
    return pairs

def metrik_hesapla(gercek_ids, tahmin_ids, tahmin_vecs,
                   gercek_yanitlar, tahmin_yanitlar):
    acc         = accuracy_score(gercek_ids, tahmin_ids)
    macro_f1    = f1_score(gercek_ids, tahmin_ids, average="macro",    zero_division=0)
    weighted_f1 = f1_score(gercek_ids, tahmin_ids, average="weighted", zero_division=0)
    f1_per_cls  = f1_score(gercek_ids, tahmin_ids, average=None,
                            zero_division=0, labels=list(range(NUM_LABELS)))
    gercek_bin  = label_binarize(gercek_ids, classes=list(range(NUM_LABELS)))
    try:
        roc_auc = roc_auc_score(gercek_bin, np.array(tahmin_vecs),
                                 average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = float("nan")
    r1, rL = [], []
    for ref, hyp in zip(gercek_yanitlar, tahmin_yanitlar):
        if ref and hyp:
            s = rouge.score(ref, hyp)
            r1.append(s["rouge1"].fmeasure)
            rL.append(s["rougeL"].fmeasure)
    return (acc, macro_f1, weighted_f1, roc_auc,
            float(np.mean(r1)) if r1 else 0.0,
            float(np.mean(rL)) if rL else 0.0,
            f1_per_cls)

def gpu_bellek_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return 0.0

# -----------------------------------------------------------------
# LORA KONFİGÜRASYONU
# -----------------------------------------------------------------
def lora_config_olustur():
    return LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                         "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM
    )

# -----------------------------------------------------------------
# ANA DÖNGÜ — 3 SEED
# -----------------------------------------------------------------
tum_sonuclar  = []
tum_log       = {}
tum_f1_per_cls = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"{MODEL_LABEL} — SEED {seed}")
    print(f"{'='*60}")

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.reset_peak_memory_stats()

    dataset = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")

    # Model yükle
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    # Eğitim
    egitim_bas = time.time()
    sft_config = SFTConfig(
        max_length=256, dataset_text_field="text",
        output_dir=f"{SAVE_PATH}/seed_{seed}/checkpoints",
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=4, per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=5, weight_decay=0.01,
        load_best_model_at_end=True, logging_steps=10,
        fp16=False, bf16=True, report_to="none", seed=seed,
    )
    trainer = SFTTrainer(
        model=model, args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config_olustur(),
    )
    trainer.train()
    egitim_sure = (time.time() - egitim_bas) / 60
    log_history  = trainer.state.log_history
    tum_log[f"seed_{seed}"] = log_history

    # Model kaydet
    seed_path = f"{SAVE_PATH}/seed_{seed}"
    os.makedirs(seed_path, exist_ok=True)
    trainer.save_model(f"{seed_path}/best_model")

    # Loss grafikleri
    train_steps  = [l["step"] for l in log_history if "loss" in l]
    train_losses = [l["loss"]      for l in log_history if "loss" in l]
    val_steps    = [l["step"]      for l in log_history if "eval_loss" in l]
    val_losses   = [l["eval_loss"] for l in log_history if "eval_loss" in l]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_steps, train_losses, label="Train Loss",
            color="#1D9E75", lw=2)
    if val_losses:
        ax.plot(val_steps, val_losses, label="Val Loss",
                color="#FF5722", lw=2, marker="o", ms=5)
    ax.set_title(f"{MODEL_LABEL} — Loss (seed {seed})",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Adım"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{seed_path}/loss_egrisi.png", dpi=150)
    plt.close()

    # Test inference
    model.eval()
    tokenizer.padding_side = "left"
    pairs = test_ciftleri_al(seed)

    gercek_ids, tahmin_ids, tahmin_vecs = [], [], []
    gercek_yanitlar, tahmin_yanitlar    = [], []
    inf_sureler = []

    for i in range(0, len(pairs), 8):
        batch     = pairs[i: i + 8]
        prompts   = [build_prompt(p[0]) for p in batch]
        gercekler = [p[1] for p in batch]
        labellar  = [p[2] for p in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=256
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=25,
                do_sample=False, pad_token_id=tokenizer.pad_token_id
            )
        inf_sureler.append((time.time() - t0) / len(batch))

        for j, out in enumerate(outputs):
            decoded = tokenizer.decode(out, skip_special_tokens=True)
            tahmin  = parse_label(decoded)
            g_id = label2id.get(labellar[j], 0)
            t_id = label2id.get(tahmin,      0)
            vec  = np.zeros(NUM_LABELS); vec[t_id] = 1.0
            gercek_ids.append(g_id); tahmin_ids.append(t_id)
            tahmin_vecs.append(vec)
            gercek_yanitlar.append(gercekler[j])
            tahmin_yanitlar.append(decoded)

    inf_ms  = np.mean(inf_sureler) * 1000
    gpu_gb  = gpu_bellek_gb()
    model_mb = sum(p.numel() * p.element_size()
                   for p in model.parameters()) / 1024**2

    (acc, macro_f1, weighted_f1, roc_auc,
     rouge1, rougeL, f1_per_cls) = metrik_hesapla(
        gercek_ids, tahmin_ids, tahmin_vecs,
        gercek_yanitlar, tahmin_yanitlar
    )
    tum_f1_per_cls.append(f1_per_cls)

    satir = {
        "Model":        MODEL_LABEL,
        "Parametre":    MODEL_PARAMS,
        "FT_Yontemi":   "LoRA",
        "Seed":         seed,
        "Accuracy":     round(acc,         4),
        "Macro_F1":     round(macro_f1,    4),
        "Weighted_F1":  round(weighted_f1, 4),
        "ROC_AUC":      round(roc_auc,     4) if not np.isnan(roc_auc) else float("nan"),
        "ROUGE_1":      round(rouge1,      4),
        "ROUGE_L":      round(rougeL,      4),
        "Egitim_dk":    round(egitim_sure, 1),
        "Inf_ms":       round(inf_ms,      1),
        "GPU_GB":       round(gpu_gb,      2),
        "Model_MB":     round(model_mb,    0),
    }
    tum_sonuclar.append(satir)

    print(f"  Acc={acc:.4f} | F1={macro_f1:.4f} | "
          f"ROUGE-1={rouge1:.4f} | GPU={gpu_gb:.2f}GB | "
          f"Egitim={egitim_sure:.1f}dk")

    # Confusion matrix (sadece seed 42)
    if seed == 42:
        cm = confusion_matrix(gercek_ids, tahmin_ids,
                              labels=list(range(NUM_LABELS)))
        fig, ax = plt.subplots(figsize=(16, 14))
        ConfusionMatrixDisplay(cm, display_labels=short_labels).plot(
            ax=ax, cmap="Greens", colorbar=True, xticks_rotation=45)
        ax.set_title(f"{MODEL_LABEL} — Confusion Matrix (seed 42)",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"{SAVE_PATH}/confusion_matrix.png", dpi=130)
        plt.close()

    # Modeli temizle
    del model, tokenizer, trainer
    gc.collect(); torch.cuda.empty_cache()

# -----------------------------------------------------------------
# ORT ± STD TABLOSU
# -----------------------------------------------------------------
df_raw  = pd.DataFrame(tum_sonuclar)
df_raw.to_csv(f"{SAVE_PATH}/ham_sonuclar.csv", index=False)

metrik_sutunlar = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC",
                   "ROUGE_1","ROUGE_L","Egitim_dk","Inf_ms","GPU_GB"]
ozet = {"Model": MODEL_LABEL, "Parametre": MODEL_PARAMS, "FT": "LoRA"}
for m in metrik_sutunlar:
    ort = df_raw[m].mean()
    std = df_raw[m].std()
    ozet[m] = f"{ort:.4f} ± {std:.4f}"
    ozet[f"{m}_ort"] = round(ort, 4)
    ozet[f"{m}_std"] = round(std, 4)

pd.DataFrame([ozet]).to_csv(f"{SAVE_PATH}/ozet_sonuclar.csv", index=False)

# -----------------------------------------------------------------
# GRAFİKLER
# -----------------------------------------------------------------
# Test metrikleri (ort ± std)
fig, ax = plt.subplots(figsize=(10, 5))
m_labels = ["Accuracy","Macro F1","Weighted F1","ROC-AUC","ROUGE-1","ROUGE-L"]
m_keys   = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC","ROUGE_1","ROUGE_L"]
m_vals   = [ozet[f"{k}_ort"] for k in m_keys]
m_stds   = [ozet[f"{k}_std"] for k in m_keys]
colors   = ["#4CAF50","#2196F3","#FF9800","#9C27B0","#E91E63","#00BCD4"]
bars = ax.bar(m_labels, m_vals, color=colors, width=0.5, edgecolor="white")
ax.errorbar(range(len(m_vals)), m_vals, yerr=m_stds,
            fmt="none", color="black", capsize=5, linewidth=1.5)
ax.set_ylim(0, 1.2); ax.set_ylabel("Skor")
ax.set_title(f"{MODEL_LABEL} — Test Metrikleri (ort ± std, 3 seed)",
             fontsize=13, fontweight="bold")
for bar, val in zip(bars, m_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/test_metrikleri.png", dpi=150)
plt.close()

# Sınıf bazlı F1 (3 seed ortalaması)
f1_ort = np.mean(tum_f1_per_cls, axis=0)
f1_std = np.std(tum_f1_per_cls, axis=0)
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(NUM_LABELS)
ax.barh(y, f1_ort, xerr=f1_std, color="#1D9E75",
        edgecolor="white", capsize=3)
ax.set_yticks(y); ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("F1 Skoru")
ax.set_title(f"{MODEL_LABEL} — Sınıf Bazlı F1 (ort ± std)",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.2)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/sinif_f1.png", dpi=150)
plt.close()

# Log kaydet
with open(f"{SAVE_PATH}/history_log.json", "w") as f:
    json.dump(tum_log, f, indent=2)

import shutil
shutil.make_archive("/content/tinyllama_output_zip", "zip", SAVE_PATH)

print(f"""
✅ FAZ 1 TAMAMLANDI — {MODEL_LABEL}
   Accuracy   : {ozet['Accuracy']}
   Macro F1   : {ozet['Macro_F1']}
   ROUGE-1    : {ozet['ROUGE_1']}
   Eğitim     : {ozet['Egitim_dk']} dk
   GPU        : {ozet['GPU_GB']} GB

   Dosyalar: /content/smollm2_output/
   ZIP      : /content/tinyllama_output_zip.zip
""")


TinyLlama-1.1B — SEED 42


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.073903,0.062595
2,0.043057,0.041853
3,0.040093,0.040431
4,0.036730,0.038656
5,0.034598,0.038608


[transformers] Both `max_new_tokens` (=25) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=25) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=25) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=25) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  Acc=0.9320 | F1=0.9282 | ROUGE-1=0.3199 | GPU=1.91GB | Egitim=11.3dk

✅ FAZ 1 TAMAMLANDI — TinyLlama-1.1B
   Accuracy   : 0.9569 ± 0.0275
   Macro F1   : 0.9549 ± 0.0291
   ROUGE-1    : 0.3217 ± 0.0022
   Eğitim     : 11.3667 ± 0.1155 dk
   GPU        : 1.9100 ± 0.0000 GB

   Dosyalar: /content/smollm2_output/
   ZIP      : /content/tinyllama_output_zip.zip



# **qwen**

In [ ]:
# =================================================================
# FAZ 1 — SmolLM2-360M LoRA Fine-tuning
# 3 seed: 42, 123, 7
# Metrikler: Acc, Macro-F1, Weighted-F1, ROC-AUC,
#            ROUGE-1, ROUGE-L, Inf (ms), GPU (GB)
# Çıktı: /content/smollm2_output/
# =================================================================

import subprocess
subprocess.run(["pip", "install", "-q", "trl", "peft", "transformers",
                "bitsandbytes", "accelerate", "rouge-score", "-U"], check=True)

import os, json, time, gc, random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
from rouge_score import rouge_scorer

# -----------------------------------------------------------------
# AYARLAR
# -----------------------------------------------------------------
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_LABEL = "Qwen2.5-1.5B"
MODEL_PARAMS = "1.5B"
DATASET_FMT  = "chatml"
SEEDS        = [42, 123, 7]
SAVE_PATH    = "/content/qwen_output"
os.makedirs(SAVE_PATH, exist_ok=True)

login(token=userdata.get("HF_TOKEN"))

# -----------------------------------------------------------------
# LABEL LİSTESİ
# -----------------------------------------------------------------
LABEL_LIST = sorted([
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
])
label2id    = {l: i for i, l in enumerate(LABEL_LIST)}
NUM_LABELS  = len(LABEL_LIST)
short_labels = [l.split("___")[-1][:18] for l in LABEL_LIST]

rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

# -----------------------------------------------------------------
# YARDIMCI FONKSİYONLAR
# -----------------------------------------------------------------
def build_prompt(sorgu):
    return (
        f"<|im_start|>system\nSen bir bitki hastalığı teşhis asistanısın. "
        f"Yanıtını 'Sınıf: <hastalık_adı>' ile başlat.<|im_end|>\n"
        f"<|im_start|>user\nBelirti: {sorgu}<|im_end|>\n"
        f"<|im_start|>assistant\nSınıf:"
    )

def parse_label(text):
    if "Sınıf:" not in text:
        return "Bilinmeyen"
    return text.split("Sınıf:")[-1].strip().split("\n")[0].strip()

def test_ciftleri_al(seed):
    ds = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")
    pairs = []
    for row in ds["test"]:
        text = row["text"]
        try:
            sorgu = text.split("<|im_start|>user\nBelirti:")[-1]\
                        .split("<|im_end|>")[0].strip()
            yanit = text.split("<|im_start|>assistant\n")[-1]\
                        .split("<|im_end|>")[0].strip()
            label = yanit.split("Sınıf:")[-1].strip().split("\n")[0].strip()
        except Exception:
            sorgu, yanit, label = text, "", "Bilinmeyen"
        pairs.append((sorgu, yanit, label))
    return pairs

def metrik_hesapla(gercek_ids, tahmin_ids, tahmin_vecs,
                   gercek_yanitlar, tahmin_yanitlar):
    acc         = accuracy_score(gercek_ids, tahmin_ids)
    macro_f1    = f1_score(gercek_ids, tahmin_ids, average="macro",    zero_division=0)
    weighted_f1 = f1_score(gercek_ids, tahmin_ids, average="weighted", zero_division=0)
    f1_per_cls  = f1_score(gercek_ids, tahmin_ids, average=None,
                            zero_division=0, labels=list(range(NUM_LABELS)))
    gercek_bin  = label_binarize(gercek_ids, classes=list(range(NUM_LABELS)))
    try:
        roc_auc = roc_auc_score(gercek_bin, np.array(tahmin_vecs),
                                 average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = float("nan")
    r1, rL = [], []
    for ref, hyp in zip(gercek_yanitlar, tahmin_yanitlar):
        if ref and hyp:
            s = rouge.score(ref, hyp)
            r1.append(s["rouge1"].fmeasure)
            rL.append(s["rougeL"].fmeasure)
    return (acc, macro_f1, weighted_f1, roc_auc,
            float(np.mean(r1)) if r1 else 0.0,
            float(np.mean(rL)) if rL else 0.0,
            f1_per_cls)

def gpu_bellek_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return 0.0

# -----------------------------------------------------------------
# LORA KONFİGÜRASYONU
# -----------------------------------------------------------------
def lora_config_olustur():
    return LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                         "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM
    )

# -----------------------------------------------------------------
# ANA DÖNGÜ — 3 SEED
# -----------------------------------------------------------------
tum_sonuclar  = []
tum_log       = {}
tum_f1_per_cls = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"{MODEL_LABEL} — SEED {seed}")
    print(f"{'='*60}")

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.reset_peak_memory_stats()

    dataset = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")

    # Model yükle
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    # Eğitim
    egitim_bas = time.time()
    sft_config = SFTConfig(
        max_length=256, dataset_text_field="text",
        output_dir=f"{SAVE_PATH}/seed_{seed}/checkpoints",
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=4, per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=5, weight_decay=0.01,
        load_best_model_at_end=True, logging_steps=10,
        fp16=False, bf16=True, report_to="none", seed=seed,
    )
    trainer = SFTTrainer(
        model=model, args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config_olustur(),
    )
    trainer.train()
    egitim_sure = (time.time() - egitim_bas) / 60
    log_history  = trainer.state.log_history
    tum_log[f"seed_{seed}"] = log_history

    # Model kaydet
    seed_path = f"{SAVE_PATH}/seed_{seed}"
    os.makedirs(seed_path, exist_ok=True)
    trainer.save_model(f"{seed_path}/best_model")

    # Loss grafikleri
    train_steps  = [l["step"] for l in log_history if "loss" in l]
    train_losses = [l["loss"]      for l in log_history if "loss" in l]
    val_steps    = [l["step"]      for l in log_history if "eval_loss" in l]
    val_losses   = [l["eval_loss"] for l in log_history if "eval_loss" in l]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_steps, train_losses, label="Train Loss",
            color="#BA7517", lw=2)
    if val_losses:
        ax.plot(val_steps, val_losses, label="Val Loss",
                color="#FF5722", lw=2, marker="o", ms=5)
    ax.set_title(f"{MODEL_LABEL} — Loss (seed {seed})",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Adım"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{seed_path}/loss_egrisi.png", dpi=150)
    plt.close()

    # Test inference
    model.eval()
    tokenizer.padding_side = "left"
    pairs = test_ciftleri_al(seed)

    gercek_ids, tahmin_ids, tahmin_vecs = [], [], []
    gercek_yanitlar, tahmin_yanitlar    = [], []
    inf_sureler = []

    for i in range(0, len(pairs), 8):
        batch     = pairs[i: i + 8]
        prompts   = [build_prompt(p[0]) for p in batch]
        gercekler = [p[1] for p in batch]
        labellar  = [p[2] for p in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=256
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=25,
                do_sample=False, pad_token_id=tokenizer.pad_token_id
            )
        inf_sureler.append((time.time() - t0) / len(batch))

        for j, out in enumerate(outputs):
            decoded = tokenizer.decode(out, skip_special_tokens=True)
            tahmin  = parse_label(decoded)
            g_id = label2id.get(labellar[j], 0)
            t_id = label2id.get(tahmin,      0)
            vec  = np.zeros(NUM_LABELS); vec[t_id] = 1.0
            gercek_ids.append(g_id); tahmin_ids.append(t_id)
            tahmin_vecs.append(vec)
            gercek_yanitlar.append(gercekler[j])
            tahmin_yanitlar.append(decoded)

    inf_ms  = np.mean(inf_sureler) * 1000
    gpu_gb  = gpu_bellek_gb()
    model_mb = sum(p.numel() * p.element_size()
                   for p in model.parameters()) / 1024**2

    (acc, macro_f1, weighted_f1, roc_auc,
     rouge1, rougeL, f1_per_cls) = metrik_hesapla(
        gercek_ids, tahmin_ids, tahmin_vecs,
        gercek_yanitlar, tahmin_yanitlar
    )
    tum_f1_per_cls.append(f1_per_cls)

    satir = {
        "Model":        MODEL_LABEL,
        "Parametre":    MODEL_PARAMS,
        "FT_Yontemi":   "LoRA",
        "Seed":         seed,
        "Accuracy":     round(acc,         4),
        "Macro_F1":     round(macro_f1,    4),
        "Weighted_F1":  round(weighted_f1, 4),
        "ROC_AUC":      round(roc_auc,     4) if not np.isnan(roc_auc) else float("nan"),
        "ROUGE_1":      round(rouge1,      4),
        "ROUGE_L":      round(rougeL,      4),
        "Egitim_dk":    round(egitim_sure, 1),
        "Inf_ms":       round(inf_ms,      1),
        "GPU_GB":       round(gpu_gb,      2),
        "Model_MB":     round(model_mb,    0),
    }
    tum_sonuclar.append(satir)

    print(f"  Acc={acc:.4f} | F1={macro_f1:.4f} | "
          f"ROUGE-1={rouge1:.4f} | GPU={gpu_gb:.2f}GB | "
          f"Egitim={egitim_sure:.1f}dk")

    # Confusion matrix (sadece seed 42)
    if seed == 42:
        cm = confusion_matrix(gercek_ids, tahmin_ids,
                              labels=list(range(NUM_LABELS)))
        fig, ax = plt.subplots(figsize=(16, 14))
        ConfusionMatrixDisplay(cm, display_labels=short_labels).plot(
            ax=ax, cmap="Oranges", colorbar=True, xticks_rotation=45)
        ax.set_title(f"{MODEL_LABEL} — Confusion Matrix (seed 42)",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"{SAVE_PATH}/confusion_matrix.png", dpi=130)
        plt.close()

    # Modeli temizle
    del model, tokenizer, trainer
    gc.collect(); torch.cuda.empty_cache()

# -----------------------------------------------------------------
# ORT ± STD TABLOSU
# -----------------------------------------------------------------
df_raw  = pd.DataFrame(tum_sonuclar)
df_raw.to_csv(f"{SAVE_PATH}/ham_sonuclar.csv", index=False)

metrik_sutunlar = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC",
                   "ROUGE_1","ROUGE_L","Egitim_dk","Inf_ms","GPU_GB"]
ozet = {"Model": MODEL_LABEL, "Parametre": MODEL_PARAMS, "FT": "LoRA"}
for m in metrik_sutunlar:
    ort = df_raw[m].mean()
    std = df_raw[m].std()
    ozet[m] = f"{ort:.4f} ± {std:.4f}"
    ozet[f"{m}_ort"] = round(ort, 4)
    ozet[f"{m}_std"] = round(std, 4)

pd.DataFrame([ozet]).to_csv(f"{SAVE_PATH}/ozet_sonuclar.csv", index=False)

# -----------------------------------------------------------------
# GRAFİKLER
# -----------------------------------------------------------------
# Test metrikleri (ort ± std)
fig, ax = plt.subplots(figsize=(10, 5))
m_labels = ["Accuracy","Macro F1","Weighted F1","ROC-AUC","ROUGE-1","ROUGE-L"]
m_keys   = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC","ROUGE_1","ROUGE_L"]
m_vals   = [ozet[f"{k}_ort"] for k in m_keys]
m_stds   = [ozet[f"{k}_std"] for k in m_keys]
colors   = ["#4CAF50","#2196F3","#FF9800","#9C27B0","#E91E63","#00BCD4"]
bars = ax.bar(m_labels, m_vals, color=colors, width=0.5, edgecolor="white")
ax.errorbar(range(len(m_vals)), m_vals, yerr=m_stds,
            fmt="none", color="black", capsize=5, linewidth=1.5)
ax.set_ylim(0, 1.2); ax.set_ylabel("Skor")
ax.set_title(f"{MODEL_LABEL} — Test Metrikleri (ort ± std, 3 seed)",
             fontsize=13, fontweight="bold")
for bar, val in zip(bars, m_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/test_metrikleri.png", dpi=150)
plt.close()

# Sınıf bazlı F1 (3 seed ortalaması)
f1_ort = np.mean(tum_f1_per_cls, axis=0)
f1_std = np.std(tum_f1_per_cls, axis=0)
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(NUM_LABELS)
ax.barh(y, f1_ort, xerr=f1_std, color="#BA7517",
        edgecolor="white", capsize=3)
ax.set_yticks(y); ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("F1 Skoru")
ax.set_title(f"{MODEL_LABEL} — Sınıf Bazlı F1 (ort ± std)",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.2)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/sinif_f1.png", dpi=150)
plt.close()

# Log kaydet
with open(f"{SAVE_PATH}/history_log.json", "w") as f:
    json.dump(tum_log, f, indent=2)

import shutil
shutil.make_archive("/content/qwen_output_zip", "zip", SAVE_PATH)

print(f"""
✅ FAZ 1 TAMAMLANDI — {MODEL_LABEL}
   Accuracy   : {ozet['Accuracy']}
   Macro F1   : {ozet['Macro_F1']}
   ROUGE-1    : {ozet['ROUGE_1']}
   Eğitim     : {ozet['Egitim_dk']} dk
   GPU        : {ozet['GPU_GB']} GB

   Dosyalar: /content/smollm2_output/
   ZIP      : /content/qwen_output_zip.zip
""")


Qwen2.5-1.5B — SEED 42


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/1168 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,0.105860,0.092036
2,0.069715,0.068249
3,0.065538,0.066683
4,0.059879,0.063591
5,0.055758,0.063582


  Acc=0.9524 | F1=0.9499 | ROUGE-1=0.4095 | GPU=3.85GB | Egitim=14.6dk

✅ FAZ 1 TAMAMLANDI — Qwen2.5-1.5B
   Accuracy   : 0.9637 ± 0.0142
   Macro F1   : 0.9620 ± 0.0151
   ROUGE-1    : 0.4078 ± 0.0059
   Eğitim     : 14.6333 ± 0.0577 dk
   GPU        : 3.8467 ± 0.0058 GB

   Dosyalar: /content/smollm2_output/
   ZIP      : /content/qwen_output_zip.zip



# **gemma**

In [ ]:
# =================================================================
# FAZ 4 — Gemma4-E2B LoRA Fine-tuning
# 3 seed: 42, 123, 7
# Metrikler: Acc, Macro-F1, Weighted-F1, ROC-AUC,
#            ROUGE-1, ROUGE-L, Inf (ms), GPU (GB)
# Çıktı: /content/gemma4_output/
#
# NOT: Gemma4 için güncel transformers gerekir
# =================================================================

import subprocess
subprocess.run(["pip", "install", "-q", "trl", "peft",
                "bitsandbytes", "accelerate", "rouge-score", "-U"], check=True)
subprocess.run(["pip", "install", "-q",
                "git+https://github.com/huggingface/transformers.git"],
               check=True)

import os, json, time, gc, random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_from_disk
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
from rouge_score import rouge_scorer

# -----------------------------------------------------------------
# AYARLAR
# -----------------------------------------------------------------
MODEL_NAME   = "google/gemma-4-E2B-it"
MODEL_LABEL  = "Gemma4-E2B"
MODEL_PARAMS = "~2B"
DATASET_FMT  = "gemma4"
SEEDS        = [42, 123, 7]
SAVE_PATH    = "/content/gemma4_output"
os.makedirs(SAVE_PATH, exist_ok=True)

login(token=userdata.get("HF_TOKEN"))

# -----------------------------------------------------------------
# LABEL LİSTESİ
# -----------------------------------------------------------------
LABEL_LIST = sorted([
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
])
label2id     = {l: i for i, l in enumerate(LABEL_LIST)}
NUM_LABELS   = len(LABEL_LIST)
short_labels = [l.split("___")[-1][:18] for l in LABEL_LIST]

rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

SISTEM = (
    "Sen bir bitki hastalığı teşhis asistanısın. "
    "Yanıtını 'Sınıf: <hastalık_adı>' ile başlat."
)

# -----------------------------------------------------------------
# YARDIMCI FONKSİYONLAR
# -----------------------------------------------------------------
def build_prompt(sorgu):
    return (
        f"<start_of_turn>user\n{SISTEM}\n\nBelirti: {sorgu}<end_of_turn>\n"
        f"<start_of_turn>model\nSınıf:"
    )

def parse_label(text):
    if "Sınıf:" not in text:
        return "Bilinmeyen"
    return text.split("Sınıf:")[-1].strip().split("\n")[0].strip()

def test_ciftleri_al(seed):
    ds = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")
    pairs = []
    for row in ds["test"]:
        text = row["text"]
        try:
            sorgu = text.split("Belirti:")[-1].split("<end_of_turn>")[0].strip()
            yanit = text.split("<start_of_turn>model\n")[-1]\
                        .split("<end_of_turn>")[0].strip()
            label = yanit.split("Sınıf:")[-1].strip().split("\n")[0].strip()
        except Exception:
            sorgu, yanit, label = text, "", "Bilinmeyen"
        pairs.append((sorgu, yanit, label))
    return pairs

def metrik_hesapla(gercek_ids, tahmin_ids, tahmin_vecs,
                   gercek_yanitlar, tahmin_yanitlar):
    acc         = accuracy_score(gercek_ids, tahmin_ids)
    macro_f1    = f1_score(gercek_ids, tahmin_ids, average="macro",    zero_division=0)
    weighted_f1 = f1_score(gercek_ids, tahmin_ids, average="weighted", zero_division=0)
    f1_per_cls  = f1_score(gercek_ids, tahmin_ids, average=None,
                            zero_division=0, labels=list(range(NUM_LABELS)))
    gercek_bin  = label_binarize(gercek_ids, classes=list(range(NUM_LABELS)))
    try:
        roc_auc = roc_auc_score(gercek_bin, np.array(tahmin_vecs),
                                 average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = float("nan")
    r1, rL = [], []
    for ref, hyp in zip(gercek_yanitlar, tahmin_yanitlar):
        if ref and hyp:
            s = rouge.score(ref, hyp)
            r1.append(s["rouge1"].fmeasure)
            rL.append(s["rougeL"].fmeasure)
    return (acc, macro_f1, weighted_f1, roc_auc,
            float(np.mean(r1)) if r1 else 0.0,
            float(np.mean(rL)) if rL else 0.0,
            f1_per_cls)

def gpu_bellek_gb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**3
    return 0.0

def lora_config_olustur():
    return LoraConfig(
        r=16, lora_alpha=32,
        target_modules="all-linear",
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM
    )

# -----------------------------------------------------------------
# ANA DÖNGÜ — 3 SEED
# -----------------------------------------------------------------
tum_sonuclar   = []
tum_log        = {}
tum_f1_per_cls = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"{MODEL_LABEL} — SEED {seed}")
    print(f"{'='*60}")

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.reset_peak_memory_stats()

    dataset = load_from_disk(f"/content/content/datasets/seed_{seed}/{DATASET_FMT}")

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    egitim_bas = time.time()
    sft_config = SFTConfig(
        max_length=256, dataset_text_field="text",
        output_dir=f"{SAVE_PATH}/seed_{seed}/checkpoints",
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=2,      # MoE için düşük
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,      # efektif batch=16
        num_train_epochs=5, weight_decay=0.01,
        load_best_model_at_end=True, logging_steps=10,
        fp16=False, bf16=True, report_to="none", seed=seed,
    )
    trainer = SFTTrainer(
        model=model, args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config_olustur(),
    )
    trainer.train()
    egitim_sure = (time.time() - egitim_bas) / 60
    log_history  = trainer.state.log_history
    tum_log[f"seed_{seed}"] = log_history

    seed_path = f"{SAVE_PATH}/seed_{seed}"
    os.makedirs(seed_path, exist_ok=True)
    trainer.save_model(f"{seed_path}/best_model")

    # Loss grafiği
    train_steps  = [l["step"] for l in log_history if "loss" in l]
    train_losses = [l["loss"]      for l in log_history if "loss" in l]
    val_steps    = [l["step"]      for l in log_history if "eval_loss" in l]
    val_losses   = [l["eval_loss"] for l in log_history if "eval_loss" in l]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_steps, train_losses, label="Train Loss",
            color="#D85A30", lw=2)
    if val_losses:
        ax.plot(val_steps, val_losses, label="Val Loss",
                color="#FF5722", lw=2, marker="o", ms=5)
    ax.set_title(f"{MODEL_LABEL} — Loss (seed {seed})",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Adım"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{seed_path}/loss_egrisi.png", dpi=150)
    plt.close()

    # Test inference
    model.eval()
    tokenizer.padding_side = "left"
    pairs = test_ciftleri_al(seed)

    gercek_ids, tahmin_ids, tahmin_vecs = [], [], []
    gercek_yanitlar, tahmin_yanitlar    = [], []
    inf_sureler = []

    for i in range(0, len(pairs), 4):   # Gemma4 için batch=4
        batch     = pairs[i: i + 4]
        prompts   = [build_prompt(p[0]) for p in batch]
        gercekler = [p[1] for p in batch]
        labellar  = [p[2] for p in batch]

        inputs = tokenizer(
            prompts, return_tensors="pt",
            padding=True, truncation=True, max_length=256
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=25,
                do_sample=False, pad_token_id=tokenizer.pad_token_id
            )
        inf_sureler.append((time.time() - t0) / len(batch))

        for j, out in enumerate(outputs):
            decoded = tokenizer.decode(out, skip_special_tokens=True)
            tahmin  = parse_label(decoded)
            g_id = label2id.get(labellar[j], 0)
            t_id = label2id.get(tahmin,      0)
            vec  = np.zeros(NUM_LABELS); vec[t_id] = 1.0
            gercek_ids.append(g_id); tahmin_ids.append(t_id)
            tahmin_vecs.append(vec)
            gercek_yanitlar.append(gercekler[j])
            tahmin_yanitlar.append(decoded)

    inf_ms   = np.mean(inf_sureler) * 1000
    gpu_gb   = gpu_bellek_gb()
    model_mb = sum(p.numel() * p.element_size()
                   for p in model.parameters()) / 1024**2

    (acc, macro_f1, weighted_f1, roc_auc,
     rouge1, rougeL, f1_per_cls) = metrik_hesapla(
        gercek_ids, tahmin_ids, tahmin_vecs,
        gercek_yanitlar, tahmin_yanitlar
    )
    tum_f1_per_cls.append(f1_per_cls)

    satir = {
        "Model":       MODEL_LABEL,
        "Parametre":   MODEL_PARAMS,
        "FT_Yontemi":  "LoRA",
        "Seed":        seed,
        "Accuracy":    round(acc,         4),
        "Macro_F1":    round(macro_f1,    4),
        "Weighted_F1": round(weighted_f1, 4),
        "ROC_AUC":     round(roc_auc,     4) if not np.isnan(roc_auc) else float("nan"),
        "ROUGE_1":     round(rouge1,      4),
        "ROUGE_L":     round(rougeL,      4),
        "Egitim_dk":   round(egitim_sure, 1),
        "Inf_ms":      round(inf_ms,      1),
        "GPU_GB":      round(gpu_gb,      2),
        "Model_MB":    round(model_mb,    0),
    }
    tum_sonuclar.append(satir)

    print(f"  Acc={acc:.4f} | F1={macro_f1:.4f} | "
          f"ROUGE-1={rouge1:.4f} | GPU={gpu_gb:.2f}GB | "
          f"Egitim={egitim_sure:.1f}dk")

    if seed == 42:
        cm = confusion_matrix(gercek_ids, tahmin_ids,
                              labels=list(range(NUM_LABELS)))
        fig, ax = plt.subplots(figsize=(16, 14))
        ConfusionMatrixDisplay(cm, display_labels=short_labels).plot(
            ax=ax, cmap="Reds", colorbar=True, xticks_rotation=45)
        ax.set_title(f"{MODEL_LABEL} — Confusion Matrix (seed 42)",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(f"{SAVE_PATH}/confusion_matrix.png", dpi=130)
        plt.close()

    del model, tokenizer, trainer
    gc.collect(); torch.cuda.empty_cache()

# -----------------------------------------------------------------
# ORT ± STD
# -----------------------------------------------------------------
df_raw = pd.DataFrame(tum_sonuclar)
df_raw.to_csv(f"{SAVE_PATH}/ham_sonuclar.csv", index=False)

metrik_sutunlar = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC",
                   "ROUGE_1","ROUGE_L","Egitim_dk","Inf_ms","GPU_GB"]
ozet = {"Model": MODEL_LABEL, "Parametre": MODEL_PARAMS, "FT": "LoRA"}
for m in metrik_sutunlar:
    ort = df_raw[m].mean()
    std = df_raw[m].std()
    ozet[m]          = f"{ort:.4f} ± {std:.4f}"
    ozet[f"{m}_ort"] = round(ort, 4)
    ozet[f"{m}_std"] = round(std, 4)

pd.DataFrame([ozet]).to_csv(f"{SAVE_PATH}/ozet_sonuclar.csv", index=False)

# Grafik
fig, ax = plt.subplots(figsize=(10, 5))
m_labels = ["Accuracy","Macro F1","Weighted F1","ROC-AUC","ROUGE-1","ROUGE-L"]
m_keys   = ["Accuracy","Macro_F1","Weighted_F1","ROC_AUC","ROUGE_1","ROUGE_L"]
m_vals   = [ozet[f"{k}_ort"] for k in m_keys]
m_stds   = [ozet[f"{k}_std"] for k in m_keys]
colors   = ["#4CAF50","#2196F3","#FF9800","#9C27B0","#E91E63","#00BCD4"]
bars = ax.bar(m_labels, m_vals, color=colors, width=0.5, edgecolor="white")
ax.errorbar(range(len(m_vals)), m_vals, yerr=m_stds,
            fmt="none", color="black", capsize=5, linewidth=1.5)
ax.set_ylim(0, 1.2); ax.set_ylabel("Skor")
ax.set_title(f"{MODEL_LABEL} — Test Metrikleri (ort ± std, 3 seed)",
             fontsize=13, fontweight="bold")
for bar, val in zip(bars, m_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
            f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")
ax.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/test_metrikleri.png", dpi=150)
plt.close()

f1_ort = np.mean(tum_f1_per_cls, axis=0)
f1_std = np.std(tum_f1_per_cls, axis=0)
fig, ax = plt.subplots(figsize=(12, 7))
y = np.arange(NUM_LABELS)
ax.barh(y, f1_ort, xerr=f1_std, color="#D85A30", edgecolor="white", capsize=3)
ax.set_yticks(y); ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel("F1 Skoru")
ax.set_title(f"{MODEL_LABEL} — Sınıf Bazlı F1 (ort ± std)",
             fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.2); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}/sinif_f1.png", dpi=150)
plt.close()

with open(f"{SAVE_PATH}/history_log.json", "w") as f:
    json.dump(tum_log, f, indent=2)

import shutil
shutil.make_archive("/content/gemma4_output_zip", "zip", SAVE_PATH)

print(f"""
✅ FAZ 4 TAMAMLANDI — {MODEL_LABEL}
   Accuracy   : {ozet['Accuracy']}
   Macro F1   : {ozet['Macro_F1']}
   ROUGE-1    : {ozet['ROUGE_1']}
   Eğitim     : {ozet['Egitim_dk']} dk
   GPU        : {ozet['GPU_GB']} GB

   Dosyalar: /content/gemma4_output/
   ZIP      : /content/gemma4_output_zip.zip
""")


Gemma4-E2B — SEED 42


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.094826,0.084076
2,0.066380,0.064637
3,0.062133,0.061432
4,0.057730,0.061352
5,0.055314,0.061445


  Acc=0.9660 | F1=0.9637 | ROUGE-1=0.3423 | GPU=15.93GB | Egitim=46.5dk

✅ FAZ 4 TAMAMLANDI — Gemma4-E2B
   Accuracy   : 0.9547 ± 0.0104
   Macro F1   : 0.9535 ± 0.0100
   ROUGE-1    : 0.3688 ± 0.0249
   Eğitim     : 46.6333 ± 0.1155 dk
   GPU        : 15.9267 ± 0.0058 GB

   Dosyalar: /content/gemma4_output/
   ZIP      : /content/gemma4_output_zip.zip



# herşeyi **ziple**

In [ ]:
!zip -r content.zip /content -x "/content/sample_data/*" "/content/drive/*"

Scanning files ......
  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/active_config (stored 0%)
  adding: content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db (deflated 97%)
  adding: content/.config/configurations/ (stored 0%)
  adding: content/.config/configurations/config_default (deflated 15%)
  adding: content/.config/default_configs.db (deflated 98%)
  adding: content/.config/logs/ (stored 0%)
  adding: content/.config/logs/2026.05.21/ (stored 0%)
  adding: content/.config/logs/2026.05.21/13.32.19.276288.log (deflated 56%)
  adding: content/.config/logs/2026.05.21/13.31.37.559893.log (deflated 92%)
  adding: content/.config/logs/2026.05.21/13.32.06.268636.log (deflated 87%)
  adding: content/.config/logs/2026.05.21/13.32.07.912587.log (deflated 58%)
  adding: content/.config/logs/2026.05.21/13.31.55.610753.log (deflated 58%)
  adding: content/.config/logs/2026.05.21/13.32.18.574065.log (deflated 57%)
  addi

In [ ]:
!mv content.zip /content/drive/MyDrive/akilliTarimOdevi/v3-801010